# Automating Deep Learning Workflows with ArcGIS API for Python
In this python notebook, we will demonstrate how to use the ArcGIS API for Python to automate deep learning tasks by executing deep learning tools on Raster Analytic Server. You will see examples of:
1. preparing training sample data with the **Export Training Data for Deep Learning** tool,
2. training a new deep learning model with the **Train Deep Learning Model** tool, and
3. using the trained model with the **Detect Objects Using Deep Learning** tool.

In [ ]:
# Import modules
import arcgis
from arcgis.gis import GIS
from arcgis.learn import export_training_data, train_model, detect_objects, Model
from arcgis.raster.analytics import list_datastore_content
from datetime import datetime

from IPython.display import IFrame

arcgis.env.verbose = True    # Print out job details during job execution

In [ ]:
# ArcGIS Enterprise Portal login
gis = GIS("https://ra-ampc.dev.geocloud.com/portal", "creator1")

In [ ]:
# Helper functions

def get_imagery_layer(gis, layer_name):
    """
    Search the Portal to find the imagery layer by the given layer_name.
    """
    # Use gis.content.search to search the input imagery Item on Portal
    search_results = gis.content.search(layer_name, item_type = "Imagery Layer")

    # Iterate through the search result to find the imagery item and get the imagery layer
    for search_result in search_results:
        if search_result.title == layer_name:
            return search_result.layers[0]

    raise Exception(f"The imagery layer: {layer_name} is not found!")

def get_feature_layer(gis, layer_name):
    """
    Search the Portal to find the feature layer by the given layer_name
    """
    # Use gis.content.search to search the input feature Item on Portal
    search_results = gis.content.search(layer_name, item_type="Feature Layer")

    # Iterate through the search result to find the feature item and get the feature layer
    for search_result in search_results:
        if search_result.title == layer_name:
            return search_result.layers[0]

    raise Exception(f"The feature layer: {layer_name} is not found!")

def get_model(gis, model_name):
    """
    Search the Portal to find the model item by the given model_name
    """
    # Use gis.content.search to search the model item on Portal
    search_results = gis.content.search(model_name, item_type="Deep Learning Package")

    # Iterate through the search result to find the model item
    for search_result in search_results:
        if search_result.title == model_name:
            return search_result

def get_current_timestamp():
    """
    Get the current timestamp as string
    """
    return datetime.now().strftime("%Y%m%d_%H%M")


## Export Training Data for Deep Learning
Documentation: https://developers.arcgis.com/python/latest/api-reference/arcgis.learn.toc.html#export-training-data
<br/>This tool converts feature (sample) data and raster data into model training data, image chips. The output will be a folder of image chips and a folder of metadata files in the specified format.

This code section performs following tasks:
1. Set the input raster, input class data, and the output location of the new job
2. Executes`export_training_data()` to submit the Export Training Data for Deep Learning job to the Raster Analytics Server on Enterprise

In [ ]:
# Use an imagery layer as the input raster
input_raster = get_imagery_layer(gis, "Redlands_NorthEast")

# Use a feature layer as the input class data (the sample features)
input_class_data = get_feature_layer(gis, "Redlands_cars")

# Use a raster store path as the output location
output_location = "/rasterStores/Raster_Cloud_Data_Store/CarChips_" + get_current_timestamp()


In [ ]:
# Display the input raster and the input class data
display_map = gis.map()                       # Create a new map object
display_map.content.add(input_raster)         # Add the input raster layer to the map
display_map.content.add(input_class_data)     # Add the input class data to the map
display_map.zoom_to_layer(input_class_data)   # Zoom to the input class data
display_map                                   # Display the map

In [ ]:
# Call export_training_data() to submit Export Training Data for Deep Learning job
export_result = export_training_data(
    input_raster = input_raster,                            # Image to be exported as training data
    input_class_data = input_class_data,                    # Training sample feature layer
    output_location = output_location,                      # Output location of the training data
    chip_format = "JPEG",                                   # Output training data format
    tile_size = {"x": 224, "y": 224},                       # Output training data size
    metadata_format = "RCNN_Masks",                         # Metadata of the training data. Determine by the model type you want to train
    context = {"parallelProcessingFactor": "50%"}           # Utilize half of processing resources to process the job
)

In [ ]:
# Display image chips output location
export_result

## Train Deep Learning Model
Documentation: https://developers.arcgis.com/python/latest/api-reference/arcgis.learn.toc.html#train-model
<br/>Train a deep learning model using the output from the Export Training Data For Deep Learning tool as the training data

This code section performs the following tasks:
1. Retrieves input training data folder path
2. Sets a pretrained model
3. Sets an output deep learning package item name
4. Executes train_model() method to submit Train Deep Learning Model job to Raster Analytics Server on Enterprise

In [ ]:
# Use the training data created by the previous export_training_data() job as the input folder
input_folder = export_result.get("uri")

# You may also use a datastore path as the input_folder:
# input_folder = "/rasterStores/Raster_Cloud_Data_Store/CarChips__20260301_1500"
# See Appendix 1 for how to use list_datastore_content() API to lookup datastore paths.


# Set the output Deep Learning Package (DLPK) name
# The output will be a DLPK item published to Portal for ArcGIS
output_name = "CarDetectionModel_" + get_current_timestamp()


# Use a DLPK item on your portal as the pretrain model
model_id = get_model(gis, "CarModel_from_scratch").id
pretrained_model = {"itemId": model_id}
# Use "Car Detection - USA" Living Atlas model as the pretrain model 
# pretrained_model = {"url": "https://www.arcgis.com/sharing/rest/content/items/cfc57b507f914d1593f5871bf0d52999"}


In [ ]:
# Call train_model() to submit Train_Deep_Learning_Model job 
train_result = train_model(
    input_folder = input_folder,             # Image chips folder
    model_type = "MASKRCNN",                 # Match pretrained model type 
    backbone_model = "RESNET50",             # Match pretrained model backbone
    batch_size = 16,                         # Number of training data being processed at once
    max_epochs = 50,                         # Number of training iterations
    output_name = output_name,               # Write model as a DLPK item on Portal for ArcGIS
    context = {"processorType":"GPU"},       # Set processor type to GPU.
    pretrained_model = pretrained_model,     # The new model will be trained based on this model
)

In [ ]:
# Display model report
model_item = get_model(gis, output_name)       # Get the model item from Portal
model = Model(model_item)                      # Initialize the model object
model_info = model.query_info()                # Get the model info
report_url = model_info.out_report.url         # Retrieve the model report URL

# Display the model report
IFrame(report_url, width=1024, height=600)


## Detect Objects Using Deep Learning
Documentation: https://developers.arcgis.com/python/latest/api-reference/arcgis.learn.toc.html#detect-objects
<br/>Use a deep learning model to find objects in the input imagery layer. The output are polygon features that mark objects the model finds.

The code section performs the following tasks:
1. Retrieves the input imagery layer from Enterprise Portal
2. Retrieves the input deep learning package from Enterprise Portal
3. Prepares model arguments for the job
4. Sets an output feature name
5. Submits a Detect Objects Using Deep Learning job to Raster Analytics Server on Enterprise


In [ ]:
# Get the input raster
input_raster = get_imagery_layer(gis, "Redlands_East")

# Prepare model
# Method 1. Use model item on Portal
model_item = get_model(gis, "CarModel_from_scratch")
model = Model(model_item)

# Method 2. Use the model trained from the previous train_model() result
# model = Model(train_result)


# Set model arguments
# Prepare model arguments
model_arguments = {
    "batch_size":16,        # How many data will be processed at once
    "threshold": 0.9,       # Objects found with over 0.9 confidence are considered as cars
    "tile_size": 224        # Width and height of image tiles into which the imagery is split for prediction
}
# More available model arguments can be found by calling Model.query_info().
# See Appendix 2 for more details.


# Set an output name
output_name = "cars_" + get_current_timestamp()

In [ ]:
# Call detect_objects() to submit Detect Objects job
detect_result = detect_objects(
    input_raster = input_raster,                                          # The image that will be used to detect objects
    model = model,                                                        # The model that will be used to run inference
    model_arguments = model_arguments,                                    # Arguments of the model
    output_name = output_name,                                            # Name of the output feature service
    max_overlap_ratio=0.2,                                                # Remove some overlapped result objects
    context = {"processorType":"GPU", "parallelProcessingFactor":"100%"}  # Use all GPU to process the job
)

In [ ]:
# Display the Object Detection result
display_map = gis.map()                    # Create a new map instance
display_map.content.add(input_raster)      # Add the input imagery layer to the map
display_map.content.add(detect_result)     # Add the result feature layer to the map
display_map.zoom_to_layer(detect_result)   # Zoom the map to the result feature layer
display_map                                # Render the map

## Appendix:
### 1. Use `list_datastore_content()` to look up datastores paths, folders, and files
Documentation: https://developers.arcgis.com/python/latest/api-reference/arcgis.raster.analytics.html#list-datastore-content

`list_datastore_content` can lookup datastore paths on your Server. <br/>
`list_datastore_content("")` will return a list of root datastore folders on your Server.<br/>
`list_datastore_content(folder_path)` will return folders and files under the given `folder_path`.

In [ ]:
from arcgis.raster.analytics import list_datastore_content

# List root datastore paths
root_lookup = list_datastore_content("")

# List child folders and files in the given datastore path
folder_path = "/rasterStores/Raster_Cloud_Data_Store/CarTrainingData"
folder_lookup = list_datastore_content(folder_path)

In [ ]:
# list_datastore_content results
print(f'list_datastore_content(""):\n{"\n".join(root_lookup)}\n')
print(f'list_datastore_content(folder_path):\n{"\n".join(folder_lookup.get(folder_path))}\n')

### 2. Use `query_info` to get model arguments
To get model arguments for Inferencing tools (`detect_objects()`, `classify_pixels()`, etc.) execution, you may use `query_info()` method. <br/>
`query_info()` sends a Query Deep Learning Model job to Raster Analytics Server on Enterprise to retrieve arguments of the model.

In [ ]:
# Get the Deep Learning Package (DLPK) item from Portal
dlpk_item = get_model(gis, "CarModel_from_scratch")

# Initialize Model object
my_model = Model(dlpk_item)

# Call query model info 
model_info = my_model.query_info()

# Print out model arguments
model_info["modelInfo"]["ParameterInfo"]
